# Mean-Reversion Strategy Example

This notebook demonstrates the Python Strategy API with a deterministic synthetic feed.

In [ ]:
from collections import deque
from pathlib import Path
import sys

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
python_dir = repo_root / "python"
if str(python_dir) not in sys.path:
    sys.path.insert(0, str(python_dir))

import pandas as pd  # noqa: E402
import backtester as backtest  # noqa: E402
from backtester import Strategy  # noqa: E402
from backtester.charts import plot_cumulative_pnl, plot_fills_on_price  # noqa: E402
from backtester.types import BookUpdate, PRICE_SCALE, Side, Trade  # noqa: E402

## Strategy

In [ ]:
class MeanReversionStrategy(Strategy):
    def __init__(self, window=3, threshold=PRICE_SCALE // 2, order_size=1):
        self.window = window
        self.threshold = threshold
        self.order_size = order_size
        self.prices = deque(maxlen=window)

    def on_book_update(self, update, ctx):
        self._on_price(
            update.instrument_id, update.price, update.size, update.timestamp_ns, ctx
        )

    def on_trade(self, trade, ctx):
        self._on_price(
            trade.instrument_id, trade.price, trade.size, trade.timestamp_ns, ctx
        )

    def _on_price(self, instrument_id, price, available_size, timestamp_ns, ctx):
        if len(self.prices) < self.window:
            self.prices.append(price)
            return

        rolling_average = sum(self.prices) // len(self.prices)
        self.prices.append(price)
        size = min(self.order_size, available_size)
        if price <= rolling_average - self.threshold:
            ctx.send_order(instrument_id, Side.BID, price, size, timestamp_ns)
        elif (
            price >= rolling_average + self.threshold
            and ctx.current_position(instrument_id) > 0
        ):
            ctx.send_order(instrument_id, Side.ASK, price, size, timestamp_ns)

## Synthetic Feed

In [ ]:
instrument_id = 10
events = [
    Trade(instrument_id, 1, 1, 100 * PRICE_SCALE, 1, Side.ASK),
    Trade(instrument_id, 2, 2, 101 * PRICE_SCALE, 1, Side.ASK),
    Trade(instrument_id, 3, 3, 102 * PRICE_SCALE, 1, Side.ASK),
    BookUpdate(instrument_id, 4, 4, Side.BID, 99 * PRICE_SCALE, 1),
    Trade(instrument_id, 5, 5, 98 * PRICE_SCALE, 1, Side.ASK),
    Trade(instrument_id, 6, 6, 103 * PRICE_SCALE, 1, Side.BID),
    BookUpdate(instrument_id, 7, 7, Side.ASK, 104 * PRICE_SCALE, 1),
]

## Run

In [ ]:
strategy = MeanReversionStrategy()
result = backtest.run(
    strategy, events=events, fill_at_touch=True, progress_interval_events=100
)
result

## Inspect Results

In [ ]:
result.pnl_series

In [ ]:
result.fills_df

In [ ]:
result.order_log_df

## Charts

In [ ]:
plot_cumulative_pnl(result)

In [ ]:
price_series = pd.Series(
    {event.timestamp_ns: getattr(event, "price") for event in events}
)
plot_fills_on_price(result, price_series)